## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**

**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno y analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


## **Resolución del Desafío 1**

Lo primero que tengo que hacer es llamar a todas las bibiotecas que necesito para realizar el desafio.

In [42]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score, accuracy_score
from sklearn.datasets import fetch_20newsgroups
import numpy as np

Lo próximo será cargar los datos que voy a utilizar como base para resolver el desafio

In [43]:
# Voy a fijar una semilla para que los resultados sean reproducibles
SEED = 19
np.random.seed(SEED)

# Creo los grupos de train y test removiendo el header, footer y quotes de los textos
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

# Instancio un vectorizador de TF-IDF
tfidfvect = TfidfVectorizer()

### Resolución punto 1

In [ ]:
# Creo un vector con los documentos de train y obtengo la dimensión del mismo
X_train = tfidfvect.fit_transform(newsgroups_train.data)
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

# Creo un vector con índices random
N_FILES = 5
rng = np.random.default_rng(SEED)
random_index = rng.integers(low=0, high=X_train.shape[0]-1, size=N_FILES)

N_CHARACTERS = 300
# Para tener como referencia, voy a imprimir las primeras lineas de los documentos seleccionados
for i in random_index:
    print("\n--------------------------------------------------")
    print(f"Documento {i}:")
    print(newsgroups_train.data[i][:N_CHARACTERS])

shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631
[ 6663  4755  4035 10474  4288]

--------------------------------------------------
Documento 6663:

I'm not saying this at all - it requires no faith on my part to
say the car drives because I've seen it drive - I've done more
than at in fact - I've actually driven it. (now what does require
some faith is the belief that my senses give an accurate representation
of what's out there....) But there

--------------------------------------------------
Documento 4755:
For Sale:

Fujitsu 324meg SCSI drive.  $450

Maxtor 338meg ESDI drive.  $425

Maxtor 160meg ESDI drive.  $225

Toshiba 106meg IDE drive.  $175

XT case & motherboard.  $50

DTC 16-bit MFM 2HD 2FD controler.  $30

All items are used, in full working condition, and have a  
warranty for one week unles

--------------------------------------------------
Documento 4035:

Secular laws seem to value criminal life mo

In [45]:
# Obtengo la similitud entre los documentos seleccionados y el resto de los documentos de train
# Voy a guardar las similitudes en una matriz de tamaño (N_FILES, cantidad de documentos de train)
similarites = np.zeros((N_FILES, X_train.shape[0]))
for i in range(N_FILES):
    similarites[i] = cosine_similarity(X_train[random_index[i]], X_train)[0]

# Voy a ordenar los vectores de similitud de mayor a menor y eliminar la primera columna que es el mismo documento
similarities_sorted = np.argsort(similarites, axis=1)[:, ::-1]
similarities_sorted = np.delete(similarities_sorted, 0, axis=1)

N_TOP = 5

# Me voy a quedar con los 5 documentos más similares a cada uno de los documentos seleccionados
top_5_similarities = similarities_sorted[:, :N_TOP]

In [46]:
# Ahora reviso si la similaridad de contenido se condice con la etiqueta de clasificación
# Para eso imprimo la categoría del documento original y la de cada uno de sus 5 más similares
for i in range(N_FILES):
    original_category = newsgroups_train.target_names[newsgroups_train.target[random_index[i]]]
    print(f"\nDocumento original {random_index[i]} - Categoría: {original_category}")
    for j in range(N_TOP):
        doc_idx = top_5_similarities[i][j]
        doc_category = newsgroups_train.target_names[newsgroups_train.target[doc_idx]]
        match = "✔" if doc_category == original_category else "✘"
        print(f"  {match} Documento {doc_idx} - Categoría: {doc_category}")


Documento original 6663 - Categoría: alt.atheism
  ✔ Documento 10194 - Categoría: alt.atheism
  ✔ Documento 5568 - Categoría: alt.atheism
  ✔ Documento 10836 - Categoría: alt.atheism
  ✔ Documento 10052 - Categoría: alt.atheism
  ✔ Documento 5200 - Categoría: alt.atheism

Documento original 4755 - Categoría: misc.forsale
  ✘ Documento 6702 - Categoría: comp.sys.ibm.pc.hardware
  ✘ Documento 3725 - Categoría: comp.sys.ibm.pc.hardware
  ✘ Documento 7722 - Categoría: comp.sys.ibm.pc.hardware
  ✘ Documento 6659 - Categoría: comp.sys.ibm.pc.hardware
  ✘ Documento 1709 - Categoría: comp.sys.ibm.pc.hardware

Documento original 4035 - Categoría: alt.atheism
  ✔ Documento 6331 - Categoría: alt.atheism
  ✔ Documento 5367 - Categoría: alt.atheism
  ✔ Documento 6410 - Categoría: alt.atheism
  ✔ Documento 80 - Categoría: alt.atheism
  ✘ Documento 8726 - Categoría: talk.politics.mideast

Documento original 10474 - Categoría: talk.politics.guns
  ✘ Documento 6437 - Categoría: talk.politics.mideast
 

**Conclusiones punto 1**

Se seleccionaron 5 documentos de manera aleatoria de las categorías `alt.atheism` (documentos 6663 y 4035), `misc.forsale` (4755), `talk.politics.guns` (10474) y `talk.politics.mideast` (4288). El patrón general es que la similaridad de coseno sobre TF-IDF funciona mejor cuando el vocabulario de la categoría es específico y poco compartido con otras categorías y falla sistemáticamente cuando varias categorías comparten un mismo campo temático:

* El documento **6663 (alt.atheism)** tiene un desempeño perfecto: sus 5 documentos más similares pertenecen exactamente a `alt.atheism`. El vocabulario típico de estos debates (ateísmo, fe, existencia de dios) es lo suficientemente particular como para no confundirse con otras categorías.

* El documento **4035 (alt.atheism)** también funciona bien (4/5 aciertos), con el único error yendo a `talk.politics.mideast`. Esto se debe a que temas religiosos y políticos de medio oriente comparten vocabulario (nacionalidad, religión, derechos), por lo que ocasionalmente se confunden entre sí.

* El caso más interesante es el documento **4755 (misc.forsale)**: sus 5 documentos más similares no coinciden ni uno con la categoría original y **los 5 caen en `comp.sys.ibm.pc.hardware`**. Mirando el contenido del documento (un anuncio de venta de discos rígidos, controladoras y motherboards con marcas y precios), tiene sentido: el vocabulario de un aviso de venta de hardware es prácticamente idéntico al de una discusión técnica sobre ese mismo hardware (nombres de marcas, modelos, tamaños en MB). La similaridad de coseno capta correctamente el **dominio del producto** (hardware de PC), pero no puede distinguir la **intención comunicativa** del texto (vender algo vs. discutirlo técnicamente), que es justamente lo que diferencia a `misc.forsale` de `comp.sys.ibm.pc.hardware`.

* El documento **10474 (talk.politics.guns)** también falla por completo: sus 5 vecinos más similares están en `talk.politics.mideast` y `talk.politics.misc`. El texto original discute "defender nuestros derechos del gobierno" en el contexto del debate sobre armas, un vocabulario (gobierno, derechos, historia) que se superpone fuertemente con otros debates políticos, aunque el tema de fondo (armas) sea distinto.

* El documento **4288 (talk.politics.mideast)** tiene un resultado mixto (2/5 aciertos), confundiéndose principalmente con `talk.politics.guns` y `alt.atheism`, nuevamente el mismo fenómeno de vocabulario político/religioso compartido.

En resumen, este muestreo confirma que la confusión entre categorías no se da únicamente entre temas "vecinos" en contenido (como religión y política), sino también entre una categoría **transaccional** (`misc.forsale`) y las categorías **temáticas** del producto que se vende (`comp.sys.ibm.pc.hardware`). Esto refuerza la idea de que la similaridad de coseno sobre TF-IDF es una medida de **cercanía de vocabulario**, no de intención o de categoría editorial y por eso puede fallar sistemáticamente en casos donde la etiqueta de clasificación depende de un criterio (como "es un aviso de venta") que no se refleja directamente en las palabras más frecuentes o distintivas del texto.

### Resolución punto 2

In [47]:
# Vectorizo los documentos de test con el mismo vectorizador ya ajustado sobre train
# (uso transform, no fit_transform, para no "ver" el vocabulario de test)
X_test = tfidfvect.transform(newsgroups_test.data)
print(f'Cantidad de documentos de test: {X_test.shape[0]}')

# Para no calcular toda la matriz de similaridad de una sola vez (7532 x 11314 en float64
# ocuparía varios cientos de MB), la calculo por lotes y en cada lote me quedo
# directamente con el documento de train más similar de cada fila
BATCH_SIZE = 500
n_test = X_test.shape[0]

# Índice, dentro de train, del documento más similar a cada documento de test
nearest_train_idx = np.zeros(n_test, dtype=int)

for start in range(0, n_test, BATCH_SIZE):
    end = min(start + BATCH_SIZE, n_test)
    batch_similarities = cosine_similarity(X_test[start:end], X_train)
    nearest_train_idx[start:end] = np.argmax(batch_similarities, axis=1)

# Asigno a cada documento de test la clase del documento de train más similar (1-NN por similaridad)
y_pred_prototype = newsgroups_train.target[nearest_train_idx]

Cantidad de documentos de test: 7532


In [48]:
# Evalúo el desempeño del modelo por prototipos con F1-score macro y con accuracy,
# para poder comparar ambas métricas y también comparar más adelante contra Naïve Bayes (punto 3)
f1_prototype = f1_score(newsgroups_test.target, y_pred_prototype, average='macro')
accuracy_prototype = accuracy_score(newsgroups_test.target, y_pred_prototype)
print(f'F1-score macro (modelo por prototipos / 1-NN por similaridad): {f1_prototype:.4f}')
print(f'Accuracy (modelo por prototipos / 1-NN por similaridad): {accuracy_prototype:.4f}')

F1-score macro (modelo por prototipos / 1-NN por similaridad): 0.5050
Accuracy (modelo por prototipos / 1-NN por similaridad): 0.5089


**Conclusiones punto 2**

El modelo por prototipos (1-NN por similaridad de coseno sobre TF-IDF) obtuvo un F1-score macro de **0.5050** y un accuracy de **0.5089** sobre el conjunto de test.

Lo primero a destacar es que ambas métricas dan valores muy parecidos entre sí. Esto es consistente con que las 20 categorías de este dataset están relativamente balanceadas en cantidad de documentos, por lo que no hay clases minoritarias que penalicen fuertemente al F1-macro por sobre el accuracy (a diferencia de lo que pasaría con un dataset muy desbalanceado). En otras palabras, acá sí es razonable decir, en términos generales, que **el modelo acierta la categoría en poco más de la mitad de los documentos de test**, y ese "poco más de la mitad" se sostiene tanto si se mide en promedio por documento (accuracy) como si se mide dándole el mismo peso a cada una de las 20 categorías (F1-macro).

Un desempeño de ~0.50 tiene bastante sentido considerando lo que se observó en el punto 1: la similaridad de coseno sobre TF-IDF captura bien el "campo temático" general de un documento, pero no logra distinguir entre categorías que comparten mucho vocabulario (como `alt.atheism` y `soc.religion.christian`, o las distintas subcategorías de `talk.politics.*`). Como este modelo no aprende ningún parámetro a partir de las etiquetas de entrenamiento (simplemente busca el vecino más parecido y le copia la clase), hereda directamente esa limitación: cualquier confusión que exista a nivel de similaridad de contenido se traduce directamente en un error de clasificación.

En resumen, este enfoque de "clasificación por prototipos" funciona como un baseline razonable y sencillo de implementar (sin necesidad de entrenar ningún modelo), pero deja bastante margen de mejora. La expectativa para el punto 3 es que un modelo que sí aprenda, a partir de los datos de entrenamiento, qué palabras son más discriminativas para cada categoría (como Naïve Bayes) logre superar este ~0.50 de F1-macro, ya que puede ponderar mejor la evidencia de todas las palabras del documento en lugar de depender de la similaridad con un único documento de entrenamiento.

### Resolución punto 3

In [49]:
# Voy a probar distintas combinaciones de parámetros del TfidfVectorizer junto con
# los dos modelos de Naïve Bayes pedidos (MultinomialNB y ComplementNB), variando también
# el alpha (suavizado) de cada modelo. NO toco ngram_range en ningún caso (queda en su
# valor por default, (1, 1)).
#
# Parámetros del vectorizador que voy a explorar:
# - stop_words: sacar o no las stopwords en inglés (reduce ruido y dimensionalidad)
# - sublinear_tf: usar 1 + log(tf) en vez de tf crudo (atenúa el efecto de palabras muy repetidas)
# - max_df / min_df: descartar términos demasiado frecuentes (poco discriminativos) o
#   demasiado raros (posible ruido/typos)

experiments = [
    {"nombre": "TFIDF default + MultinomialNB (alpha=1.0)",
     "vect_params": {}, "modelo": MultinomialNB, "alpha": 1.0},

    {"nombre": "TFIDF default + ComplementNB (alpha=1.0)",
     "vect_params": {}, "modelo": ComplementNB, "alpha": 1.0},

    {"nombre": "TFIDF sin stopwords + MultinomialNB (alpha=1.0)",
     "vect_params": {"stop_words": "english"}, "modelo": MultinomialNB, "alpha": 1.0},

    {"nombre": "TFIDF sin stopwords + ComplementNB (alpha=1.0)",
     "vect_params": {"stop_words": "english"}, "modelo": ComplementNB, "alpha": 1.0},

    {"nombre": "TFIDF sin stopwords + sublinear_tf + MultinomialNB (alpha=0.1)",
     "vect_params": {"stop_words": "english", "sublinear_tf": True}, "modelo": MultinomialNB, "alpha": 0.1},

    {"nombre": "TFIDF sin stopwords + sublinear_tf + ComplementNB (alpha=0.1)",
     "vect_params": {"stop_words": "english", "sublinear_tf": True}, "modelo": ComplementNB, "alpha": 0.1},

    {"nombre": "TFIDF sin stopwords + sublinear_tf + min_df/max_df + ComplementNB (alpha=0.1)",
     "vect_params": {"stop_words": "english", "sublinear_tf": True, "min_df": 2, "max_df": 0.5},
     "modelo": ComplementNB, "alpha": 0.1},
]

resultados = []

for exp in experiments:
    vectorizer = TfidfVectorizer(**exp["vect_params"])
    X_train_exp = vectorizer.fit_transform(newsgroups_train.data)
    X_test_exp = vectorizer.transform(newsgroups_test.data)

    modelo = exp["modelo"](alpha=exp["alpha"])
    modelo.fit(X_train_exp, newsgroups_train.target)
    y_pred_exp = modelo.predict(X_test_exp)

    f1_exp = f1_score(newsgroups_test.target, y_pred_exp, average="macro")
    resultados.append({"nombre": exp["nombre"], "f1_macro": f1_exp})

In [50]:
# Imprimo una tabla comparativa ordenada de mejor a peor F1-macro
resultados_ordenados = sorted(resultados, key=lambda r: r["f1_macro"], reverse=True)

print(f"{'Experimento':<70} {'F1-macro'}")
print("-" * 80)
for r in resultados_ordenados:
    print(f"{r['nombre']:<70} {r['f1_macro']:.4f}")

mejor = resultados_ordenados[0]
print(f"\nMejor combinación: {mejor['nombre']} -> F1-macro = {mejor['f1_macro']:.4f}")

Experimento                                                            F1-macro
--------------------------------------------------------------------------------
TFIDF sin stopwords + ComplementNB (alpha=1.0)                         0.6936
TFIDF default + ComplementNB (alpha=1.0)                               0.6930
TFIDF sin stopwords + sublinear_tf + ComplementNB (alpha=0.1)          0.6900
TFIDF sin stopwords + sublinear_tf + min_df/max_df + ComplementNB (alpha=0.1) 0.6878
TFIDF sin stopwords + sublinear_tf + MultinomialNB (alpha=0.1)         0.6715
TFIDF sin stopwords + MultinomialNB (alpha=1.0)                        0.6468
TFIDF default + MultinomialNB (alpha=1.0)                              0.5854

Mejor combinación: TFIDF sin stopwords + ComplementNB (alpha=1.0) -> F1-macro = 0.6936


**Conclusiones punto 3**

La mejor combinación encontrada fue **TF-IDF sin stopwords + ComplementNB (alpha=1.0)**, con un F1-macro de **0.6936**. Comparado contra el modelo por prototipos del punto 2 (F1-macro ≈ 0.505), esto representa una mejora sustancial (+0.19), lo cual tiene sentido: a diferencia del 1-NN por similaridad, Naïve Bayes sí aprende, a partir de todos los documentos de entrenamiento, qué palabras son más probables en cada categoría, en lugar de depender pura y exclusivamente de un único documento "más parecido".

Algunas observaciones puntuales sobre los experimentos:

* **ComplementNB superó siempre a MultinomialNB**, manteniendo el resto de los parámetros iguales (0.6936 vs 0.6468 sin stopwords, 0.6930 vs 0.5854 con TF-IDF default). Esto es esperable: ComplementNB fue diseñado específicamente para corregir los supuestos poco realistas que hace MultinomialNB (que asume que todas las clases generan documentos con una distribución de palabras similar), y en la práctica suele funcionar mejor en clasificación de texto, incluso cuando las clases están razonablemente balanceadas como en este dataset.

* **Sacar las stopwords ayudó mucho a MultinomialNB** (de 0.5854 a 0.6468), pero apenas modificó a ComplementNB (0.6930 → 0.6936). Esto sugiere que ComplementNB ya es bastante más robusto al ruido de palabras muy frecuentes y poco discriminativas, mientras que MultinomialNB se ve más afectado por ellas.

* **`sublinear_tf` combinado con un alpha más chico (0.1) ayudó notablemente a MultinomialNB** (0.6468 → 0.6715), pero **empeoró levemente a ComplementNB** (0.6936 → 0.6900). Atenuar los términos muy repetidos (usando 1 + log(tf) en vez de tf crudo) parece compensar mejor las limitaciones de MultinomialNB que las de ComplementNB, que ya maneja bien esa situación por su propia formulación.

* **Filtrar términos muy frecuentes o muy raros (`min_df`/`max_df`) empeoró levemente el resultado** (0.6900 → 0.6878 en ComplementNB). Esto indica que, en este dataset, incluso las palabras poco frecuentes aportan señal útil para distinguir entre las 20 categorías, y descartarlas quita más información de la que quita ruido.

En resumen, para este problema el factor que más impactó en el desempeño fue **la elección del modelo** (ComplementNB por sobre MultinomialNB) más que los ajustes finos del vectorizador. Esto es consistente con lo que documenta la propia literatura de scikit-learn: ComplementNB suele ser una mejor alternativa por default para clasificación de texto que MultinomialNB.

### Resolución punto 4

In [51]:
# Transpongo la matriz documento-término (X_train es documentos x vocabulario) para
# obtener una matriz término-documento (vocabulario x documentos). Ahora cada fila
# representa una palabra como un vector, en función de en qué documentos aparece y
# con qué peso TF-IDF. Esto me permite tratar la similaridad entre palabras de la
# misma forma que hice entre documentos en el punto 1.
X_terms = X_train.T.tocsr()
feature_names = tfidfvect.get_feature_names_out()
print(f'Shape de la matriz término-documento: {X_terms.shape}')
print(f'Cantidad de palabras (vocabulario): {X_terms.shape[0]}')
print(f'Cantidad de documentos: {X_terms.shape[1]}')

Shape de la matriz término-documento: (101631, 11314)
Cantidad de palabras (vocabulario): 101631
Cantidad de documentos: 11314


In [52]:
# Elijo manualmente 5 palabras interpretables, cada una asociada a una temática
# distinta de las que aparecen en 20 Newsgroups, y busco sus 5 palabras más similares
WORDS = ["car", "baseball", "computer", "god", "space"]

N_TOP_WORDS = 5

for word in WORDS:
    word_idx = tfidfvect.vocabulary_[word]

    # Similaridad de la palabra elegida contra todas las palabras del vocabulario
    word_similarities = cosine_similarity(X_terms[word_idx], X_terms)[0]

    # Ordeno de mayor a menor similaridad y descarto la propia palabra (similaridad = 1)
    top_indices = np.argsort(word_similarities)[::-1]
    top_indices = top_indices[top_indices != word_idx][:N_TOP_WORDS]

    print(f"\nPalabra: '{word}'")
    for idx in top_indices:
        print(f"  {feature_names[idx]:<20} (similaridad: {word_similarities[idx]:.4f})")


Palabra: 'car'
  cars                 (similaridad: 0.1797)
  criterium            (similaridad: 0.1770)
  civic                (similaridad: 0.1748)
  owner                (similaridad: 0.1689)
  dealer               (similaridad: 0.1681)

Palabra: 'baseball'
  tommorrow            (similaridad: 0.1839)
  football             (similaridad: 0.1759)
  penna                (similaridad: 0.1734)
  wintry               (similaridad: 0.1690)
  espn                 (similaridad: 0.1677)

Palabra: 'computer'
  decwriter            (similaridad: 0.1563)
  deluged              (similaridad: 0.1522)
  harkens              (similaridad: 0.1522)
  shopper              (similaridad: 0.1443)
  the                  (similaridad: 0.1361)

Palabra: 'god'
  jesus                (similaridad: 0.2688)
  bible                (similaridad: 0.2616)
  that                 (similaridad: 0.2560)
  existence            (similaridad: 0.2548)
  christ               (similaridad: 0.2511)

Palabra: 'space'
  nasa  

**Conclusiones punto 4**

Los resultados son bastante disparejos entre palabras y hay un patrón claro: **funciona mucho mejor con palabras de vocabulario "técnico" y acotado a un tema muy específico** que con palabras genéricas que aparecen en muchísimos documentos de distintas temáticas.

* **`space`** y **`god`** dan, por lejos, los resultados más interpretables y con mayor similaridad (0.25 ~ 0.33): `space` trae `nasa`, `shuttle`, `seti` (todo vocabulario específico de la temática espacial/astronomía) y `god` trae `jesus`, `bible`, `christ` (vocabulario religioso cristiano). Tiene sentido: son palabras que aparecen casi exclusivamente en un conjunto acotado de documentos de una temática puntual, por lo que su vector (definido por en qué documentos aparece) queda muy concentrado y se superpone naturalmente con otras palabras de esa misma temática.

* **`car`** da un resultado intermedio, mayormente interpretable (`cars`, `civic` (un modelo de auto), `owner`, `dealer`), aunque aparece `criterium` (una carrera ciclística) que no tiene relación semántica evidente: probablemente sea una coincidencia de que ambas palabras aparecen juntas en un puñado de documentos puntuales.

* **`baseball`** y sobre todo **`computer`** dan los resultados más ruidosos. `baseball` sí trae `football` y `espn` (coherentes), pero también `tommorrow`, `penna` y `wintry`, que parecen residuos de un post puntual sobre un partido pasado (comentando el clima o la fecha) más que una relación temática real. `computer` es el peor caso: `decwriter`, `deluged`, `harkens`, `shopper` e incluso la palabra `the` no tienen relación semántica clara con "computadora".

**¿Por qué pasa esto?** A diferencia de la similaridad entre documentos del punto 1 (donde cada vector tiene miles de palabras no nulas y el ruido puntual se diluye), acá cada palabra es un vector de solo 11314 posiciones (una por documento) y palabras muy genéricas como `computer` o `baseball` aparecen en cientos de documentos de contextos completamente distintos. Esto hace que su vector sea "difuso": no se concentra en ningún conjunto chico de documentos, y su similaridad de coseno con el resto de las palabras queda dominada por coincidencias puntuales de co-ocurrencia en unos pocos documentos, en vez de reflejar una relación semántica real. Además, en general los valores de similaridad obtenidos acá (0.13–0.33) son notoriamente más bajos que los que se veían entre documentos en el punto 1, lo cual refuerza que la señal semántica capturada es mucho más débil.

En definitiva, este experimento muestra una limitación conocida de usar TF-IDF transpuesta como "vector de palabras": el esquema fue diseñado para representar documentos (ponderando qué tan discriminativa es una palabra *dentro* de un documento), no para capturar relaciones semánticas *entre* palabras. Funciona razonablemente bien como aproximación cuando la palabra pertenece a un campo semántico muy acotado (`space`, `god`), pero se degrada con palabras de uso más general. Para obtener vectores de palabras semánticamente más consistentes en todos los casos haría falta un método pensado específicamente para eso, como Word2Vec o GloVe, que aprenden las representaciones optimizando directamente por relaciones de co-ocurrencia en contexto, en vez de derivarlas como subproducto de una matriz pensada para otro fin.